# Introspection Experiments — Colab Runner

This notebook runs the full introspection experiment pipeline on Colab GPUs.

**Pipeline:**
1. Setup & install dependencies
2. Generate steering vectors (or use pre-computed ones)
3. Run intervention experiments (control + steering)
4. Download results

**Requirements:** Colab Pro+ with A100 GPU recommended.

## 1. Check GPU & Setup

In [ ]:
!nvidia-smi
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Clone repo and install dependencies
# Note: Colab doesn't have Python 3.13 or uv, so we install deps directly
# and add src/ to the path instead of doing `pip install -e .`
# Skip Git LFS — steering vectors are downloaded from HuggingFace Hub instead
!GIT_LFS_SKIP_SMUDGE=1 git clone https://github.com/agastyasridharan/introspection.git
%cd introspection

# Install deps (torch is already pre-installed on Colab, so skip it)
!pip install -q accelerate transformers huggingface-hub tqdm 2>&1 | tail -3

# Add src/ to Python path so `from introspection import ...` works
import sys
sys.path.insert(0, "src")
print(f"Python {sys.version}")
print("Setup complete")

In [ ]:
# Verify the package imports correctly
from introspection import generate_steering_vectors, steer, hooks
print("Package imported successfully")

## 2. Configuration

Choose the model by setting `MODEL_NAME` — layer indices, batch sizes, and data paths auto-configure for cross-model comparison.

| Model | VRAM (bf16) | GPU Required |
|-------|-------------|-------------|
| Qwen3-8B | ~16GB | T4 / A100 |
| Qwen3-14B | ~28GB | A100 40GB |
| Qwen3-32B | ~64GB | A100 80GB |
| Qwen3-235B-A22B (FP8) | ~120GB | Multi-GPU / A100 x2 |

In [ ]:
# ===== EDIT THIS =====

MODEL_NAME = "Qwen/Qwen3-8B"
# Options:
#   "Qwen/Qwen3-8B"                              — ~16GB, T4 / A100
#   "Qwen/Qwen3-14B"                             — ~28GB, A100 40GB+
#   "Qwen/Qwen3-32B"                             — ~64GB, A100 80GB
#   "Qwen/Qwen3-235B-A22B-Instruct-2507-FP8"     — ~120GB FP8, multi-GPU

CONCEPT_COUNT = 50                   # Number of concepts (max 50)
SEED = 13                           # Random seed for reproducibility
STRENGTHS = [3.5, 4.0, 4.5, 5.0, 6.0] # Steering vector multipliers
TEMPERATURES = [0.7]                    # Sampling temperatures
TRIALS = 5                             # Trials per configuration (more = better stats)

# ===== AUTO-CONFIGURED FROM MODEL_NAME =====
# Override any of these after this block if needed.

from introspection.constants import MODEL_LAYER_COUNTS

# Per-model defaults: data dir short name, dtype, and batch sizes.
# Generation batches are small (KV cache grows per token); logit batches
# can be much larger (single forward pass, no KV growth).
_MODEL_DEFAULTS = {
    "Qwen/Qwen3-8B":                             {"short": "8b",      "dtype": "bfloat16", "gen_batch": 10, "logit_batch": 50},
    "Qwen/Qwen3-14B":                            {"short": "14b",     "dtype": "bfloat16", "gen_batch": 6,  "logit_batch": 35},
    "Qwen/Qwen3-32B":                            {"short": "32b",     "dtype": "bfloat16", "gen_batch": 3,  "logit_batch": 15},
    "Qwen/Qwen3-235B-A22B-Instruct-2507-FP8":    {"short": "235a22b", "dtype": "bfloat16", "gen_batch": 2,  "logit_batch": 8},
}

_defaults = _MODEL_DEFAULTS[MODEL_NAME]
DTYPE = _defaults["dtype"]
MAX_BATCH_SIZE = _defaults["gen_batch"]
LOGIT_MAX_BATCH_SIZE = _defaults["logit_batch"]

# Compute layer indices at matched relative positions for cross-model comparison.
# Targets: ~14%, ~29%, ~43%, ~57%, ~71%, ~86%, 100% of model depth.
# These match the 8B experiment's [5,10,15,20,25,30,35] exactly.
total_layers = MODEL_LAYER_COUNTS[MODEL_NAME]
_max_layer = total_layers - 1
_LAYER_FRACTIONS = [1/7, 2/7, 3/7, 4/7, 5/7, 6/7, 1.0]
LAYERS = sorted(set(round(f * _max_layer) for f in _LAYER_FRACTIONS))

# Paths
DATA_DIR = f"data/qwen_{_defaults['short']}"
STEERING_VECTOR_PATH = f"{DATA_DIR}/steering_vectors.pt"
SWEEP_OUTPUT_PATH = f"{DATA_DIR}/sweep_colab.json"
LOGIT_OUTPUT_PATH = f"{DATA_DIR}/logit_experiment.json"
INVERTED_LOGIT_OUTPUT_PATH = f"{DATA_DIR}/inverted_logit_experiment.json"

# Validate
invalid = [l for l in LAYERS if l >= total_layers]
if invalid:
    raise ValueError(f"Layers {invalid} exceed model's {total_layers} layers")

print(f"Model: {MODEL_NAME} ({total_layers} layers)")
print(f"Dtype: {DTYPE}")
print(f"Layers: {LAYERS}")
print(f"  → layer %: {[round(l / _max_layer * 100, 1) for l in LAYERS]}")
print(f"Strengths: {STRENGTHS}")
print(f"Trials: {TRIALS}")
print(f"Batch sizes: generation={MAX_BATCH_SIZE}, logit={LOGIT_MAX_BATCH_SIZE}")
print(f"Data dir: {DATA_DIR}")

## 3. Load Steering Vectors

Downloads pre-computed steering vectors from HuggingFace Hub (~5 seconds).
Falls back to generating from scratch if the download fails (~10 minutes on A100).

Skip this cell if you already have `steering_vectors.pt` from a previous run.

In [ ]:
import os
from pathlib import Path

sv_path = Path(STEERING_VECTOR_PATH)
if sv_path.exists() and sv_path.stat().st_size > 1000:
    print(f"Steering vectors already exist at {sv_path} ({sv_path.stat().st_size / 1e6:.1f} MB)")
    print("Skipping. Delete the file to re-download or regenerate.")
else:
    try:
        from huggingface_hub import hf_hub_download
        print(f"Downloading steering vectors for {MODEL_NAME}...")
        sv_path.parent.mkdir(parents=True, exist_ok=True)
        hf_hub_download(
            repo_id="agastyasridharan/introspection-steering-vectors",
            filename=f"qwen_{_defaults['short']}/steering_vectors.pt",
            local_dir="data",
            repo_type="dataset",
        )
        print(f"Downloaded! ({sv_path.stat().st_size / 1e6:.1f} MB)")
    except Exception as e:
        print(f"Download failed ({e}), generating from scratch...")
        from introspection.generate_steering_vectors import run_experiment
        sv_path.parent.mkdir(parents=True, exist_ok=True)
        run_experiment(
            model_name=MODEL_NAME,
            dtype_name=DTYPE,
            output_path=sv_path,
            concept_count=CONCEPT_COUNT,
            seed=SEED,
        )
        print(f"Generated! ({sv_path.stat().st_size / 1e6:.1f} MB)")


## 4. Run Intervention Experiments

This is the main experiment: for each (concept × layer × strength × trial), generate a control response (no steering) and an intervention response (with steering vector injected).

**Runtime estimate:** ~1-3 hours for 50 concepts × 7 layers × 5 strengths × 5 trials on A100.

In [ ]:
from pathlib import Path
from introspection.steer import (
    load_steering_vectors, prepare_prompt, steer as run_steer,
    PROMPT_MESSAGES,
)
from introspection.types import ExperimentArgs
from introspection.utils import load_model, resolve_torch_dtype
import json

# Reload model and steering vectors if the model has changed since last run.
_need_model = True
_need_vectors = True
try:
    if _loaded_model_name == MODEL_NAME:
        print(f"Reusing model already on {model.device}")
        _need_model = False
except NameError:
    pass
try:
    if _loaded_sv_path == STEERING_VECTOR_PATH:
        print(f"Reusing {len(steering_vectors)} steering vectors")
        _need_vectors = False
except NameError:
    pass

if _need_model:
    print(f"Loading {MODEL_NAME}...")
    tokenizer, model = load_model(
        model_name=MODEL_NAME,
        dtype=resolve_torch_dtype(DTYPE),
        disable_cache=False,
        set_pad_token_to_eos=True,
    )
    _loaded_model_name = MODEL_NAME
    print(f"Model loaded on {model.device}")

if _need_vectors:
    steering_vectors = load_steering_vectors(Path(STEERING_VECTOR_PATH))
    _loaded_sv_path = STEERING_VECTOR_PATH
    print(f"Loaded {len(steering_vectors)} concepts")

concept_names = sorted(steering_vectors.keys())
print(f"Loaded {len(concept_names)} concepts: {concept_names[:5]}...")

# Prepare prompt
template_prompt = prepare_prompt(tokenizer, model.device)

# Build experiment args
args = ExperimentArgs(
    model_name=MODEL_NAME,
    dtype_name=DTYPE,
    steering_vector_path=Path(STEERING_VECTOR_PATH),
    concepts=None,  # use all
    layers=LAYERS,
    strengths=STRENGTHS,
    json_path=Path(SWEEP_OUTPUT_PATH),
    temperatures=TEMPERATURES,
    top_p=0.8,
    top_k=20,
    min_p=0.0,
    trials=TRIALS,
    max_new_tokens=200,
    do_sample=True,
    seed=SEED,
    debug_residual=False,
    max_batch_size=MAX_BATCH_SIZE,
)

# Run!
print(f"\n=== Running {len(concept_names)} concepts × {len(LAYERS)} layers × {len(STRENGTHS)} strengths × {TRIALS} trials ===")
records = run_steer(
    args=args,
    concept_names=concept_names,
    all_steering_vectors=steering_vectors,
    tokenizer=tokenizer,
    model=model,
    template_prompt=template_prompt,
)

# Save results
output_path = Path(SWEEP_OUTPUT_PATH)
output_path.parent.mkdir(parents=True, exist_ok=True)
experiment_summary = {
    "model_name": MODEL_NAME,
    "steering_vector_path": str(args.steering_vector_path),
    "dtype": DTYPE,
    "prompt": {
        "messages": PROMPT_MESSAGES,
        "formatted": template_prompt.formatted_prompt,
        "injection_index": template_prompt.injection_index,
    },
    "settings": {
        "strengths": STRENGTHS,
        "temperatures": TEMPERATURES,
        "top_p": args.top_p,
        "top_k": args.top_k,
        "min_p": args.min_p,
        "max_new_tokens": args.max_new_tokens,
        "trials": TRIALS,
        "do_sample": args.do_sample,
        "seed": SEED,
        "layers": LAYERS,
        "concepts_requested": None,
        "max_batch_size": MAX_BATCH_SIZE,
    },
    "concepts_evaluated": concept_names,
    "results": records,
}
with output_path.open("w", encoding="utf-8") as f:
    json.dump(experiment_summary, f, ensure_ascii=False, indent=2)
    f.write("\n")

print(f"\nSaved {len(records)} records to {output_path}")

## 5. Download Results

Download the sweep JSON to your local machine. You can then commit it to your repo and run grading (Stage 3) and visualization (Stage 4) locally.

In [ ]:
from google.colab import files
import os

# Download the sweep results
sweep_path = SWEEP_OUTPUT_PATH
if os.path.exists(sweep_path):
    size_mb = os.path.getsize(sweep_path) / 1e6
    print(f"Downloading {sweep_path} ({size_mb:.1f} MB)...")
    files.download(sweep_path)
else:
    print(f"No file at {sweep_path} — run the experiment first")

# Also download steering vectors if they were freshly generated
sv_path = STEERING_VECTOR_PATH
if os.path.exists(sv_path):
    size_mb = os.path.getsize(sv_path) / 1e6
    print(f"\nSteering vectors at {sv_path} ({size_mb:.1f} MB)")
    # Uncomment to download:
    # files.download(sv_path)

## 6. Quick Sanity Check

Preview a few results before downloading.

In [ ]:
import json, random

with open(SWEEP_OUTPUT_PATH) as f:
    data = json.load(f)

results = data["results"]
print(f"Total records: {len(results)}")
print(f"Concepts: {len(data['concepts_evaluated'])}")
print(f"Layers: {sorted(set(str(r['layers']) for r in results))}")
print(f"Strengths: {sorted(set(r['strength'] for r in results))}")
print(f"Trials: {sorted(set(r['trial'] for r in results))}")

# Show a few random examples
print("\n" + "="*80)
for r in random.sample(results, min(3, len(results))):
    print(f"\nConcept: {r['concept']} | Layer: {r['layers']} | Strength: {r['strength']}")
    print(f"  Control:      {r['control'][:150]}...")
    print(f"  Intervention: {r['intervention'][:150]}...")

---

## 7. Logit-Based Introspection Experiment (NEW)

This is a cleaner version of the experiment that uses **logit extraction** instead of free-form generation + LLM grading. It runs a **2x2 design**: detection vs factual control questions, with and without steering injection.

- **Detection prompt**: "Did you detect an injected thought?" (with steering)
- **Factual control**: "Can humans breathe underwater?" (with same steering)

If the model shows elevated YES-logits for detection but NOT for factual questions, that's evidence for genuine introspection rather than a general YES-bias.

~200x faster than the generation-based experiment (single forward pass per batch).

In [ ]:
# All logit experiment config (LOGIT_OUTPUT_PATH, LOGIT_MAX_BATCH_SIZE)
# is set in the main config cell above. Just confirm:
print(f"Model: {MODEL_NAME}")
print(f"Layers: {LAYERS}")
print(f"Strengths: {STRENGTHS}")
print(f"Logit batch size: {LOGIT_MAX_BATCH_SIZE}")
print(f"Output: {LOGIT_OUTPUT_PATH}")

In [ ]:
from pathlib import Path
from introspection.steer import load_steering_vectors, set_random_seed
from introspection.logit_steer import run_logit_experiment
from introspection.types import LogitExperimentArgs
from introspection.utils import load_model, resolve_torch_dtype
import json

# Reload model and steering vectors if the model has changed since last run.
# (Avoids silently reusing 14B vectors when switching to 32B, etc.)
_need_model = True
_need_vectors = True
try:
    if _loaded_model_name == MODEL_NAME:
        print(f"Reusing model already on {model.device}")
        _need_model = False
except NameError:
    pass
try:
    if _loaded_sv_path == STEERING_VECTOR_PATH:
        print(f"Reusing {len(steering_vectors)} steering vectors")
        _need_vectors = False
except NameError:
    pass

if _need_model:
    print(f"Loading {MODEL_NAME}...")
    tokenizer, model = load_model(
        model_name=MODEL_NAME,
        dtype=resolve_torch_dtype(DTYPE),
        disable_cache=False,
        set_pad_token_to_eos=True,
    )
    _loaded_model_name = MODEL_NAME
    print(f"Model loaded on {model.device}")

if _need_vectors:
    steering_vectors = load_steering_vectors(Path(STEERING_VECTOR_PATH))
    _loaded_sv_path = STEERING_VECTOR_PATH
    print(f"Loaded {len(steering_vectors)} concepts")

set_random_seed(SEED)

args = LogitExperimentArgs(
    model_name=MODEL_NAME,
    dtype_name=DTYPE,
    steering_vector_path=Path(STEERING_VECTOR_PATH),
    concepts=None,
    layers=LAYERS,
    strengths=STRENGTHS,
    json_path=Path(LOGIT_OUTPUT_PATH),
    seed=SEED,
    debug_residual=False,
    max_batch_size=LOGIT_MAX_BATCH_SIZE,
)

output = run_logit_experiment(
    args=args,
    model=model,
    tokenizer=tokenizer,
    all_steering_vectors=steering_vectors,
)

# Save
output_path = Path(LOGIT_OUTPUT_PATH)
output_path.parent.mkdir(parents=True, exist_ok=True)
with output_path.open("w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)
    f.write("\n")

n_detection = sum(1 for r in output["results"] if r["condition"] == "detection")
n_factual = sum(1 for r in output["results"] if r["condition"] == "factual")
print(f"\nSaved {len(output['results'])} records ({n_detection} detection, {n_factual} factual)")
print(f"  to {output_path}")

### Quick Analysis: Detection vs Factual Control (COMMENTED OUT)

This analysis cell is preserved for future reference. Uncomment the code cell below to run it. It groups logit experiment results by (layer, strength) and prints mean detection shift, factual shift, and introspection score for each configuration.

In [ ]:
# [COMMENTED OUT] Quick Analysis: Detection vs Factual Control
# Uncomment everything below to run the logit analysis inline.
#
# import json
# import numpy as np
#
# with open(LOGIT_OUTPUT_PATH) as f:
#     data = json.load(f)
#
# results = data["results"]
# detection_baseline = data["baselines"]["detection_no_injection"]["logit_diff"]
#
# # Separate detection and factual results
# det = [r for r in results if r["condition"] == "detection"]
# fac = [r for r in results if r["condition"] == "factual"]
#
# print(f"Detection baseline (no injection): logit_diff = {detection_baseline:.3f}")
# print(f"{'':>20} {'Detection':>12} {'Factual':>12} {'Introspection':>14}")
# print("-" * 62)
#
# # Group by (layer, strength) and compute mean logit_diff
# from collections import defaultdict
# det_by_config = defaultdict(list)
# fac_by_config = defaultdict(list)
#
# for r in det:
#     det_by_config[(r["layer"], r["strength"])].append(r["logit_diff"])
# for r in fac:
#     fac_by_config[(r["layer"], r["strength"])].append(r["logit_diff"])
#
# for (layer, strength) in sorted(det_by_config.keys()):
#     d_mean = np.mean(det_by_config[(layer, strength)])
#     f_mean = np.mean(fac_by_config[(layer, strength)])
#     introspection = d_mean - f_mean
#     print(f"  L{layer:>3} S{strength:>4.1f}    {d_mean:>+10.3f}   {f_mean:>+10.3f}   {introspection:>+12.3f}")

### Download Logit Experiment Results

In [ ]:
from google.colab import files
import os

if os.path.exists(LOGIT_OUTPUT_PATH):
    size_mb = os.path.getsize(LOGIT_OUTPUT_PATH) / 1e6
    print(f"Downloading {LOGIT_OUTPUT_PATH} ({size_mb:.1f} MB)...")
    files.download(LOGIT_OUTPUT_PATH)
else:
    print(f"No file at {LOGIT_OUTPUT_PATH} — run the logit experiment first")

---

## 7b. Inverted Logit Experiment

Same framework as the logit experiment, but the detection question is **inverted**: instead of asking
"Did you detect an injected thought?" (YES = detected), it asks "Was this a control trial (no injection)?"
(NO = detected). Factual controls are also inverted to use unambiguous YES-answer questions.

If the model has genuine introspective access, it should shift toward NO (correct) when injected.
If it merely has a YES-bias from off-distribution perturbation, it will shift toward YES — the wrong answer.

Combined with Section 7, this cleanly separates semantic introspection from surface-level token bias.

In [ ]:
from pathlib import Path
from introspection.steer import load_steering_vectors, set_random_seed
from introspection.logit_steer import run_logit_experiment
from introspection.types import LogitExperimentArgs
from introspection.utils import load_model, resolve_torch_dtype
import json

# Reload model and steering vectors if the model has changed since last run.
_need_model = True
_need_vectors = True
try:
    if _loaded_model_name == MODEL_NAME:
        print(f"Reusing model already on {model.device}")
        _need_model = False
except NameError:
    pass
try:
    if _loaded_sv_path == STEERING_VECTOR_PATH:
        print(f"Reusing {len(steering_vectors)} steering vectors")
        _need_vectors = False
except NameError:
    pass

if _need_model:
    print(f"Loading {MODEL_NAME}...")
    tokenizer, model = load_model(
        model_name=MODEL_NAME,
        dtype=resolve_torch_dtype(DTYPE),
        disable_cache=False,
        set_pad_token_to_eos=True,
    )
    _loaded_model_name = MODEL_NAME
    print(f"Model loaded on {model.device}")

if _need_vectors:
    steering_vectors = load_steering_vectors(Path(STEERING_VECTOR_PATH))
    _loaded_sv_path = STEERING_VECTOR_PATH
    print(f"Loaded {len(steering_vectors)} concepts")

set_random_seed(SEED)

args = LogitExperimentArgs(
    model_name=MODEL_NAME,
    dtype_name=DTYPE,
    steering_vector_path=Path(STEERING_VECTOR_PATH),
    concepts=None,
    layers=LAYERS,
    strengths=STRENGTHS,
    json_path=Path(INVERTED_LOGIT_OUTPUT_PATH),
    seed=SEED,
    debug_residual=False,
    max_batch_size=LOGIT_MAX_BATCH_SIZE,
    inverted=True,
)

output = run_logit_experiment(
    args=args,
    model=model,
    tokenizer=tokenizer,
    all_steering_vectors=steering_vectors,
)

# Save
output_path = Path(INVERTED_LOGIT_OUTPUT_PATH)
output_path.parent.mkdir(parents=True, exist_ok=True)
with output_path.open("w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)
    f.write("\n")

n_detection = sum(1 for r in output["results"] if r["condition"] == "detection")
n_factual = sum(1 for r in output["results"] if r["condition"] == "factual")
print(f"\nSaved {len(output['results'])} records ({n_detection} detection, {n_factual} factual)")
print(f"  to {output_path}")

### Download Inverted Logit Experiment Results

In [ ]:
from google.colab import files
import os

if os.path.exists(INVERTED_LOGIT_OUTPUT_PATH):
    size_mb = os.path.getsize(INVERTED_LOGIT_OUTPUT_PATH) / 1e6
    print(f"Downloading {INVERTED_LOGIT_OUTPUT_PATH} ({size_mb:.1f} MB)...")
    files.download(INVERTED_LOGIT_OUTPUT_PATH)
else:
    print(f"No file at {INVERTED_LOGIT_OUTPUT_PATH} — run the inverted logit experiment first")

---

## 8. Concept Mismatch Experiment

This experiment tests whether the model can discriminate *which* concept was injected, not just whether *any* injection occurred. It does this by injecting concept Y's steering vector while asking the model about concept X.

**Three conditions per (named concept X, layer, strength):**

| Condition | Prompt names | Vector injected | If introspection | If YES-bias |
|-----------|-------------|-----------------|-----------------|-------------|
| Congruent | X | X | YES | YES |
| Incongruent | X | Y (distant) | NO | YES |
| Baseline | X | none | NO | NO |

For each concept, K=5 maximally dissimilar partners are selected by cosine similarity of steering vectors at the reference layer. If the model shows higher YES-logits for congruent than incongruent, that's evidence for concept-specific detection. If congruent ≈ incongruent > baseline, it's just generic perturbation bias.

**Runtime estimate:** ~10-20 minutes for 50 concepts × 7 layers × 5 strengths on A100.

In [ ]:
MISMATCH_OUTPUT_PATH = f"{DATA_DIR}/mismatch_experiment.json"
K_PARTNERS = 5  # dissimilar partners per concept

print(f"Model: {MODEL_NAME}")
print(f"Layers: {LAYERS}")
print(f"Strengths: {STRENGTHS}")
print(f"K partners: {K_PARTNERS}")
print(f"Logit batch size: {LOGIT_MAX_BATCH_SIZE}")
print(f"Output: {MISMATCH_OUTPUT_PATH}")

In [ ]:
from pathlib import Path
from introspection.steer import load_steering_vectors, set_random_seed
from introspection.mismatch_steer import run_mismatch_experiment
from introspection.types import MismatchExperimentArgs
from introspection.utils import load_model, resolve_torch_dtype
import json

# Reload model and steering vectors if needed
_need_model = True
_need_vectors = True
try:
    if _loaded_model_name == MODEL_NAME:
        print(f"Reusing model already on {model.device}")
        _need_model = False
except NameError:
    pass
try:
    if _loaded_sv_path == STEERING_VECTOR_PATH:
        print(f"Reusing {len(steering_vectors)} steering vectors")
        _need_vectors = False
except NameError:
    pass

if _need_model:
    print(f"Loading {MODEL_NAME}...")
    tokenizer, model = load_model(
        model_name=MODEL_NAME,
        dtype=resolve_torch_dtype(DTYPE),
        disable_cache=False,
        set_pad_token_to_eos=True,
    )
    _loaded_model_name = MODEL_NAME
    print(f"Model loaded on {model.device}")

if _need_vectors:
    steering_vectors = load_steering_vectors(Path(STEERING_VECTOR_PATH))
    _loaded_sv_path = STEERING_VECTOR_PATH
    print(f"Loaded {len(steering_vectors)} concepts")

set_random_seed(SEED)

# Reference layer = middle of LAYERS
ref_layer = LAYERS[len(LAYERS) // 2]

args = MismatchExperimentArgs(
    model_name=MODEL_NAME,
    dtype_name=DTYPE,
    steering_vector_path=Path(STEERING_VECTOR_PATH),
    concepts=None,  # all concepts
    layers=LAYERS,
    strengths=STRENGTHS,
    json_path=Path(MISMATCH_OUTPUT_PATH),
    seed=SEED,
    debug_residual=False,
    max_batch_size=LOGIT_MAX_BATCH_SIZE,
    k_partners=K_PARTNERS,
    reference_layer=ref_layer,
)

output = run_mismatch_experiment(
    args=args,
    model=model,
    tokenizer=tokenizer,
    all_steering_vectors=steering_vectors,
)

# Save
output_path = Path(MISMATCH_OUTPUT_PATH)
output_path.parent.mkdir(parents=True, exist_ok=True)
with output_path.open("w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)
    f.write("\n")

n_cong = sum(1 for r in output["results"] if r["condition"] == "congruent")
n_incong = sum(1 for r in output["results"] if r["condition"] == "incongruent")
n_base = sum(1 for r in output["results"] if r["condition"] == "baseline")
print(f"\nSaved {len(output['results'])} records ({n_cong} congruent, {n_incong} incongruent, {n_base} baseline)")
print(f"  to {output_path}")

### Quick Analysis: Congruent vs Incongruent vs Baseline

In [ ]:
import json
import numpy as np
from collections import defaultdict

with open(MISMATCH_OUTPUT_PATH) as f:
    data = json.load(f)

results = data["results"]
baselines = data["baselines_by_concept"]

# Separate by condition
cong = [r for r in results if r["condition"] == "congruent"]
incong = [r for r in results if r["condition"] == "incongruent"]
base = [r for r in results if r["condition"] == "baseline"]

# --- Overall summary ---
cong_diffs = [r["logit_diff"] for r in cong]
incong_diffs = [r["logit_diff"] for r in incong]
base_diffs = [r["logit_diff"] for r in base]

print("=" * 70)
print("OVERALL SUMMARY (all concepts, layers, strengths)")
print("=" * 70)
print(f"{'Condition':<15} {'Mean logit_diff':>16} {'Std':>10} {'N':>6}")
print("-" * 50)
print(f"{'Congruent':<15} {np.mean(cong_diffs):>+16.3f} {np.std(cong_diffs):>10.3f} {len(cong_diffs):>6}")
print(f"{'Incongruent':<15} {np.mean(incong_diffs):>+16.3f} {np.std(incong_diffs):>10.3f} {len(incong_diffs):>6}")
print(f"{'Baseline':<15} {np.mean(base_diffs):>+16.3f} {np.std(base_diffs):>10.3f} {len(base_diffs):>6}")
print()
gap = np.mean(cong_diffs) - np.mean(incong_diffs)
print(f"Congruent - Incongruent gap: {gap:+.3f}")
print(f"  → {'Positive = concept-specific detection signal' if gap > 0 else 'Negative or zero = no concept discrimination'}")

# --- By layer ---
print("\n" + "=" * 70)
print("BY LAYER (averaged across concepts and strengths)")
print("=" * 70)
print(f"{'Layer':>6} {'Congruent':>12} {'Incongruent':>12} {'Baseline':>12} {'Cong-Incong':>12}")
print("-" * 58)

cong_by_layer = defaultdict(list)
incong_by_layer = defaultdict(list)
base_by_layer = defaultdict(list)
for r in cong:
    cong_by_layer[r["layer"]].append(r["logit_diff"])
for r in incong:
    incong_by_layer[r["layer"]].append(r["logit_diff"])
for r in base:
    base_by_layer[r["layer"]].append(r["logit_diff"])

for layer in sorted(cong_by_layer.keys()):
    cm = np.mean(cong_by_layer[layer])
    im = np.mean(incong_by_layer[layer])
    bm = np.mean(base_by_layer[layer])
    print(f"{layer:>6} {cm:>+12.3f} {im:>+12.3f} {bm:>+12.3f} {cm - im:>+12.3f}")

# --- By strength ---
print("\n" + "=" * 70)
print("BY STRENGTH (averaged across concepts and layers)")
print("=" * 70)
print(f"{'Strength':>8} {'Congruent':>12} {'Incongruent':>12} {'Baseline':>12} {'Cong-Incong':>12}")
print("-" * 60)

cong_by_str = defaultdict(list)
incong_by_str = defaultdict(list)
for r in cong:
    cong_by_str[r["strength"]].append(r["logit_diff"])
for r in incong:
    incong_by_str[r["strength"]].append(r["logit_diff"])

for strength in sorted(cong_by_str.keys()):
    cm = np.mean(cong_by_str[strength])
    im = np.mean(incong_by_str[strength])
    print(f"{strength:>8.1f} {cm:>+12.3f} {im:>+12.3f} {'':>12} {cm - im:>+12.3f}")

# --- Per-concept discrimination scores ---
print("\n" + "=" * 70)
print("PER-CONCEPT DISCRIMINATION (congruent - mean incongruent, across all configs)")
print("=" * 70)

concept_cong = defaultdict(list)
concept_incong = defaultdict(list)
for r in cong:
    concept_cong[r["named_concept"]].append(r["logit_diff"])
for r in incong:
    concept_incong[r["named_concept"]].append(r["logit_diff"])

disc_scores = {}
for concept in sorted(concept_cong.keys()):
    cm = np.mean(concept_cong[concept])
    im = np.mean(concept_incong[concept])
    disc_scores[concept] = cm - im

# Sort by discrimination score
sorted_concepts = sorted(disc_scores.items(), key=lambda x: x[1], reverse=True)
print(f"\nTop 10 (highest discrimination):")
for concept, score in sorted_concepts[:10]:
    print(f"  {concept:20s} {score:+.3f}")
print(f"\nBottom 10 (lowest discrimination):")
for concept, score in sorted_concepts[-10:]:
    print(f"  {concept:20s} {score:+.3f}")

n_positive = sum(1 for _, s in sorted_concepts if s > 0)
print(f"\nConcepts with positive discrimination: {n_positive}/{len(sorted_concepts)}")

### Download Mismatch Experiment Results

In [ ]:
from google.colab import files
import os

if os.path.exists(MISMATCH_OUTPUT_PATH):
    size_mb = os.path.getsize(MISMATCH_OUTPUT_PATH) / 1e6
    print(f"Downloading {MISMATCH_OUTPUT_PATH} ({size_mb:.1f} MB)...")
    files.download(MISMATCH_OUTPUT_PATH)
else:
    print(f"No file at {MISMATCH_OUTPUT_PATH} — run the mismatch experiment first")